# Testing 2-Pass GP Baseline on March 18 Bundle

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import sys
import warnings
warnings.filterwarnings('ignore')

candidate_roots = [Path.cwd(), *Path.cwd().parents]
repo_root = next(
    (p for p in candidate_roots if (p / "malca").is_dir() and (p / "malca" / "baseline.py").exists()),
    Path.cwd(),
)

for path in (repo_root, repo_root / "malca"):
    sp = str(path.resolve())
    if sp not in sys.path:
        sys.path.insert(0, sp)

from malca.lightcurve_io import load_lightcurve_df
from malca.baseline import per_camera_gp_baseline_masked
from malca.config import GP_DIP_SIGMA_THRESH, GP_BRIGHT_SIGMA_THRESH
from malca.notebook_paths import discover_bundled_lightcurve_paths


In [ ]:
RUN_NAME = 'runs_march18_bundle_all'
LIGHTCURVE_DIR = repo_root / 'output' / 'runs' / RUN_NAME / 'bundle_assets' / 'lightcurves'
MAX_FILES = 200

if LIGHTCURVE_DIR.is_dir():
    files = [str(p) for p in sorted(LIGHTCURVE_DIR.glob('*.dat3'))[:MAX_FILES]]
else:
    files = [str(p) for p in discover_bundled_lightcurve_paths(repo_root / 'output' / 'runs', limit=MAX_FILES)]

print(f'Using {len(files)} bundled light curves from {LIGHTCURVE_DIR if LIGHTCURVE_DIR.is_dir() else repo_root / "output" / "runs"}')


In [ ]:
target_ids = ['635656111241', '489626721133', '438086746412', '214748665650']
target_files = [str(LIGHTCURVE_DIR / f'{t}.dat3') for t in target_ids]
plot_files = target_files + [f for f in files if f not in target_files][:1]

for file in plot_files:
    df = load_lightcurve_df(file)
    if df is None or len(df) == 0:
        continue
        
    from malca.lightcurve_io import to_asassn_algorithm_frame
    df = to_asassn_algorithm_frame(df)
    df = per_camera_gp_baseline_masked(df)
    
    # Create a 2x2 grid: Top-Left=Stage 1 (Stiff GP), Top-Right=Stage 2 (Masking),
    # Bottom-Left=Stage 3 (Consensus), Bottom-Right=Stage 4 (Final GP)
    fig, axes = plt.subplots(2, 2, figsize=(20, 12), sharex=True, sharey=True)
    fig.suptitle(f'4-Stage GP Baseline for {Path(file).name}', fontsize=16)
    
    ax1, ax2, ax3, ax4 = axes.flatten()
    
    for cam, sub in df.groupby('camera#'):
        sub_sorted = sub.sort_values('JD')
        
        # Stage 1: Initial Stiff GP
        ax1.errorbar(sub['JD'], sub['mag'], yerr=sub['error'], fmt='.', alpha=0.3, label=f'Cam {cam} data')
        if 'base_rough' in sub_sorted.columns and not sub_sorted['base_rough'].isna().all():
            ax1.plot(sub_sorted['JD'], sub_sorted['base_rough'], '-', linewidth=2, label=f'Cam {cam} base_rough')
            
        # Stage 2: Masking
        good = ~sub['is_masked'].fillna(False).astype(bool)
        bad = ~good
        ax2.errorbar(sub.loc[good, 'JD'], sub.loc[good, 'mag'], yerr=sub.loc[good, 'error'], fmt='.', alpha=0.3)
        if bad.any():
            ax2.errorbar(sub.loc[bad, 'JD'], sub.loc[bad, 'mag'], yerr=sub.loc[bad, 'error'], fmt='x', color='red', alpha=0.7, label=f'Cam {cam} MASKED')
        if 'base_rough' in sub_sorted.columns and not sub_sorted['base_rough'].isna().all():
            ax2.plot(sub_sorted['JD'], sub_sorted['base_rough'], '--', linewidth=1, alpha=0.5)
            
        # Stage 3: Consensus (if applicable)
        ax3.errorbar(sub['JD'], sub['mag'], yerr=sub['error'], fmt='.', alpha=0.3)
        if 'base_consensus' in sub_sorted.columns and not sub_sorted['base_consensus'].isna().all():
            ax3.plot(sub_sorted['JD'], sub_sorted['base_consensus'], '-', linewidth=2, label=f'Cam {cam} consensus')
        elif 'base_rough' in sub_sorted.columns and not sub_sorted['base_rough'].isna().all():
            ax3.plot(sub_sorted['JD'], sub_sorted['base_rough'], '--', linewidth=1, alpha=0.5, label=f'Cam {cam} no consensus (uses base_rough)')
            
        # Stage 4: Final Flexible GP
        ax4.errorbar(sub['JD'], sub['mag'], yerr=sub['error'], fmt='.', alpha=0.3)
        if 'baseline' in sub_sorted.columns and not sub_sorted['baseline'].isna().all():
            ax4.plot(sub_sorted['JD'], sub_sorted['baseline'], '-', linewidth=2, label=f'Cam {cam} final baseline')

    ax1.invert_yaxis()
    
    ax1.set_title('Stage 1: Initial Stiff GP (base_rough)')
    ax1.legend()
    
    ax2.set_title('Stage 2: Excursion Masking')
    ax2.legend()
    
    ax3.set_title('Stage 3: Anchor Consensus')
    ax3.legend()
    
    ax4.set_title('Stage 4: Final Flexible GP')
    ax4.legend()
    
    plt.tight_layout()
    plt.show()
